# 03 — SQL Feature Engineering

**Objective.** Use the project SQL pipeline to convert raw technology-stock OHLCV observations into finance-ready modeling tables.

This notebook focuses on the analytical layer that sits between raw market data and Bayesian modeling. The core outputs are:

- cleaned adjusted-close price histories;
- daily simple and log returns;
- rolling volatility, momentum, moving-average, and liquidity features;
- drawdown and downside-risk metrics;
- annualized stock-level summaries; and
- an aligned wide return panel for portfolio modeling.

The notebook also validates SQL calculations against independent Python calculations for `AAPL` and writes the processed modeling dataset to `data/processed/tech_stock_features.csv`.


## 1. Why SQL is useful for financial analytics

SQL is a strong fit for repeatable financial feature engineering because it expresses the transformation from raw observations to analytical tables declaratively:

- **Reproducible transformations.** Versioned SQL scripts define exactly how each table or view is built. Re-running the same scripts against the same raw data produces the same cleaned prices, returns, and risk features.
- **Window functions.** Financial features are naturally ordered by `symbol` and `date`. SQL window functions such as `LAG`, `AVG(...) OVER (...)`, `STDDEV_SAMP(...) OVER (...)`, and running `MAX(...) OVER (...)` make return, rolling-volatility, moving-average, and drawdown calculations concise and auditable.
- **Auditability.** Each intermediate table (`clean_prices`, `daily_returns`, `rolling_features`, `drawdowns`) can be inspected directly. This makes it easier to trace a model input back to the raw adjusted close and volume records that produced it.
- **Scalable feature generation.** Once the logic is written, the same SQL can process many symbols and many dates without writing symbol-specific Python loops. DuckDB executes these analytical queries efficiently on local files while preserving a warehouse-style workflow.

The most important finance convention in this notebook is that **return calculations use adjusted close (`adj_close`)**, not raw close. Adjusted close accounts for splits and other corporate actions, so it is the appropriate price series for return and risk modeling.


## 2. Imports and project paths

The notebook can be run from the repository root or from the `notebooks/` directory. The path setup below ensures local imports from `src/` work in both cases.


In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

# Resolve the repository root whether this notebook is launched from the repo
# root or from the notebooks/ directory.
NOTEBOOK_PATH = Path.cwd().resolve()
ROOT = NOTEBOOK_PATH if (NOTEBOOK_PATH / "src").exists() else NOTEBOOK_PATH.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config import DUCKDB_PATH, PROCESSED_FEATURE_PATH, RAW_CSV_PATH, SQL_DIR
from src.sql_utils import initialize_database, query_to_df, run_sql_pipeline

pd.set_option("display.max_columns", 80)
pd.set_option("display.float_format", "{:.6f}".format)

# In this repository snapshot, the source CSV may be available either at the
# configured production path or at the repository root. Prefer the configured
# path, but fall back to the checked-in sample CSV for a self-contained run.
RAW_CSV_FOR_NOTEBOOK = RAW_CSV_PATH if RAW_CSV_PATH.exists() else ROOT / "tech_stocks.csv"

print(f"Repository root: {ROOT}")
print(f"Raw CSV used:     {RAW_CSV_FOR_NOTEBOOK}")
print(f"DuckDB path:      {DUCKDB_PATH}")
print(f"SQL directory:    {SQL_DIR}")
print(f"Feature output:   {PROCESSED_FEATURE_PATH}")


## 3. Run the full SQL pipeline

The standard entry point is `src.sql_utils.initialize_database`, which loads the raw CSV into DuckDB and creates `clean_prices`. The full feature-engineering pipeline is then executed with `src.sql_utils.run_sql_pipeline`.

The pipeline scripts run in this order:

1. `sql/02_clean_prices.sql`
2. `sql/03_daily_returns.sql`
3. `sql/04_rolling_features.sql`
4. `sql/05_risk_metrics.sql`
5. `sql/06_portfolio_inputs.sql`


In [ ]:
con = initialize_database(raw_csv_path=RAW_CSV_FOR_NOTEBOOK, db_path=DUCKDB_PATH)
run_sql_pipeline(con)

pipeline_objects = [
    "clean_prices",
    "daily_returns",
    "rolling_features",
    "drawdowns",
    "stock_level_risk_summary",
    "annual_stock_metrics",
    "portfolio_returns_wide",
]

row_counts = []
for object_name in pipeline_objects:
    row_count = con.execute(f"SELECT COUNT(*) FROM {object_name}").fetchone()[0]
    row_counts.append({"sql_object": object_name, "rows": row_count})

pd.DataFrame(row_counts)


## 4. Inspect SQL output samples

Each table or view below is an intermediate or final analytical asset. Inspecting samples at this stage is a practical audit step before using the data in models.


In [ ]:
def show_sql_sample(table_name: str, order_by: str = "1", limit: int = 8) -> pd.DataFrame:
    """Display a small ordered sample from a SQL table or view."""
    display(Markdown(f"### `{table_name}`"))
    sample = query_to_df(
        con,
        f"""
        SELECT *
        FROM {table_name}
        ORDER BY {order_by}
        LIMIT {limit}
        """,
    )
    display(sample)
    return sample

samples = {
    "clean_prices": show_sql_sample("clean_prices", "symbol, date"),
    "daily_returns": show_sql_sample("daily_returns", "symbol, date"),
    "rolling_features": show_sql_sample("rolling_features", "symbol, date"),
    "drawdowns": show_sql_sample("drawdowns", "symbol, date"),
    "stock_level_risk_summary": show_sql_sample("stock_level_risk_summary", "symbol"),
    "annual_stock_metrics": show_sql_sample("annual_stock_metrics", "symbol, year"),
    "portfolio_returns_wide": show_sql_sample("portfolio_returns_wide", "date"),
}


## 5. SQL logic for finance features

This section documents the key SQL transformations. The snippets mirror the versioned SQL files in `sql/` and are shown here so the notebook is self-explanatory.


### 5.1 Log returns

Daily returns are created from adjusted close prices. The SQL uses `LAG(adj_close)` partitioned by `symbol` and ordered by `date` to find the previous trading day's adjusted close.

```sql
WITH lagged_prices AS (
    SELECT
        symbol,
        date,
        adj_close,
        volume,
        LAG(adj_close) OVER (
            PARTITION BY symbol
            ORDER BY date
        ) AS previous_adj_close
    FROM clean_prices
)
SELECT
    symbol,
    date,
    adj_close,
    volume,
    (adj_close / previous_adj_close) - 1 AS simple_return,
    LN(adj_close / previous_adj_close) AS log_return,
    adj_close * volume AS dollar_volume
FROM lagged_prices
WHERE previous_adj_close IS NOT NULL;
```

Log returns are additive across time and are therefore convenient for Bayesian return models and portfolio return panels.


In [ ]:
query_to_df(
    con,
    """
    SELECT symbol, date, adj_close, simple_return, log_return, dollar_volume
    FROM daily_returns
    WHERE symbol = 'AAPL'
    ORDER BY date
    LIMIT 10
    """,
)


### 5.2 Rolling volatility

Rolling volatility is calculated with SQL window frames. The project stores daily rolling standard deviations and annualized versions using `SQRT(252)`.

```sql
STDDEV_SAMP(simple_return) OVER (
    PARTITION BY symbol
    ORDER BY date
    ROWS BETWEEN 20 PRECEDING AND CURRENT ROW
) AS raw_rolling_vol_21d

CASE WHEN return_count_21d = 21 THEN raw_rolling_vol_21d END AS rolling_vol_21d,
rolling_vol_21d * SQRT(252) AS rolling_ann_vol_21d
```

The `CASE WHEN return_count_21d = 21` guard prevents partial-window statistics from being treated as full 21-trading-day estimates.


In [ ]:
query_to_df(
    con,
    """
    SELECT
        symbol,
        date,
        simple_return,
        rolling_vol_21d,
        rolling_ann_vol_21d,
        rolling_vol_63d,
        rolling_ann_vol_63d
    FROM rolling_features
    WHERE symbol = 'AAPL'
      AND rolling_vol_21d IS NOT NULL
    ORDER BY date
    LIMIT 10
    """,
)


### 5.3 Drawdowns

Drawdown measures the percentage decline from a prior running peak. SQL computes the running peak with a cumulative `MAX(adj_close)` window.

```sql
SELECT
    symbol,
    date,
    adj_close,
    MAX(adj_close) OVER (
        PARTITION BY symbol
        ORDER BY date
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS running_peak,
    (adj_close / MAX(adj_close) OVER (
        PARTITION BY symbol
        ORDER BY date
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    )) - 1 AS drawdown
FROM clean_prices;
```

The most negative drawdown for a symbol is its maximum drawdown over the observed period.


In [ ]:
query_to_df(
    con,
    """
    SELECT symbol, date, adj_close, running_peak, drawdown
    FROM drawdowns
    WHERE symbol = 'AAPL'
    ORDER BY drawdown ASC, date
    LIMIT 10
    """,
)


### 5.4 Bad-day indicators

Downside-risk features count how often a stock has large negative daily returns. The SQL converts threshold events into `1.0` / `0.0` indicators and averages them into rates.

```sql
AVG(CASE WHEN simple_return <= -0.02 THEN 1.0 ELSE 0.0 END) AS bad_day_rate_2pct,
AVG(CASE WHEN simple_return <= -0.05 THEN 1.0 ELSE 0.0 END) AS bad_day_rate_5pct
```

These features are useful because two assets can have similar volatility but very different frequencies of severe negative days.


In [ ]:
query_to_df(
    con,
    """
    SELECT
        symbol,
        trading_days,
        worst_daily_return,
        bad_day_rate_2pct,
        bad_day_rate_5pct
    FROM stock_level_risk_summary
    ORDER BY bad_day_rate_2pct DESC, symbol
    """,
)


### 5.5 Annualized metrics

The SQL layer annualizes mean daily log returns and daily volatility using a 252-trading-day convention.

```sql
AVG(log_return) * 252 AS annualized_return,
STDDEV_SAMP(log_return) * SQRT(252) AS annualized_volatility,
CASE
    WHEN annualized_volatility > 0
        THEN annualized_return / annualized_volatility
END AS sharpe_like_ratio
```

The ratio is labeled `sharpe_like_ratio` because it does not subtract a risk-free rate; it is a return-per-unit-volatility diagnostic rather than a full Sharpe ratio.


### 5.6 Tail-risk, Sharpe-like, and diversification conventions

The same SQL features support the project-wide finance calculations:

- Annualized return uses `AVG(log_return) * 252`; annualized volatility uses `STDDEV_SAMP(log_return) * SQRT(252)`.
- The drawdown table computes the running peak with a cumulative `MAX(adj_close)` window, then stores current drawdown as `adj_close / running_peak - 1`; maximum drawdown is the minimum drawdown by stock.
- Historical 5% VaR is the empirical 5th percentile of daily returns, interpreted as the loss threshold exceeded on the worst 5% of days. Historical 5% CVaR averages returns at or below that threshold, so it describes expected loss conditional on already being in the left tail.
- Sharpe-like ratios use `(annualized_return - risk_free_rate) / annualized_volatility` with `risk_free_rate = 0` for simplicity; they are not formal Sharpe ratios unless a risk-free return is included.
- Portfolio correlations are computed only after aligning stocks on common dates. Highly correlated tech stocks limit diversification because shared growth, rates, and sentiment exposures can dominate stock-specific differences in stress periods.



In [ ]:
query_to_df(
    con,
    """
    SELECT
        symbol,
        year,
        trading_days,
        annualized_return,
        annualized_volatility,
        sharpe_like_ratio,
        max_drawdown_in_year,
        bad_day_rate_2pct
    FROM annual_stock_metrics
    WHERE symbol = 'AAPL'
    ORDER BY year
    """,
)


## 6. Validate SQL outputs against Python calculations for `AAPL`

The checks below recompute the most important features in pandas from `clean_prices` and compare them with the SQL tables. This independent calculation helps catch SQL window-frame mistakes, adjusted-close mistakes, and annualization mistakes before the features feed downstream models.


In [ ]:
SYMBOL_TO_VALIDATE = "AAPL"
TOLERANCE = 1e-10

clean_aapl = query_to_df(
    con,
    f"""
    SELECT symbol, date, adj_close, volume
    FROM clean_prices
    WHERE symbol = '{SYMBOL_TO_VALIDATE}'
    ORDER BY date
    """,
)
clean_aapl["date"] = pd.to_datetime(clean_aapl["date"])

sql_daily = query_to_df(
    con,
    f"""
    SELECT date, simple_return, log_return, dollar_volume
    FROM daily_returns
    WHERE symbol = '{SYMBOL_TO_VALIDATE}'
    ORDER BY date
    """,
)
sql_daily["date"] = pd.to_datetime(sql_daily["date"])

sql_rolling = query_to_df(
    con,
    f"""
    SELECT date, rolling_vol_21d, rolling_ann_vol_21d
    FROM rolling_features
    WHERE symbol = '{SYMBOL_TO_VALIDATE}'
    ORDER BY date
    """,
)
sql_rolling["date"] = pd.to_datetime(sql_rolling["date"])

sql_drawdowns = query_to_df(
    con,
    f"""
    SELECT date, running_peak, drawdown
    FROM drawdowns
    WHERE symbol = '{SYMBOL_TO_VALIDATE}'
    ORDER BY date
    """,
)
sql_drawdowns["date"] = pd.to_datetime(sql_drawdowns["date"])

sql_risk_summary = query_to_df(
    con,
    f"""
    SELECT *
    FROM stock_level_risk_summary
    WHERE symbol = '{SYMBOL_TO_VALIDATE}'
    """,
)

# Independent Python calculations.
py_prices = clean_aapl.copy()
py_prices["previous_adj_close"] = py_prices["adj_close"].shift(1)
py_prices["py_simple_return"] = py_prices["adj_close"].pct_change()
py_prices["py_log_return"] = np.log(py_prices["adj_close"] / py_prices["previous_adj_close"])
py_prices["py_dollar_volume"] = py_prices["adj_close"] * py_prices["volume"]
py_prices["py_running_peak"] = py_prices["adj_close"].cummax()
py_prices["py_drawdown"] = (py_prices["adj_close"] / py_prices["py_running_peak"]) - 1

py_daily = py_prices.dropna(subset=["previous_adj_close"])[
    ["date", "py_simple_return", "py_log_return", "py_dollar_volume"]
]
py_daily["py_rolling_vol_21d"] = py_daily["py_simple_return"].rolling(21).std(ddof=1)
py_daily["py_rolling_ann_vol_21d"] = py_daily["py_rolling_vol_21d"] * np.sqrt(252)

joined_daily = sql_daily.merge(py_daily, on="date", how="inner")
joined_rolling = sql_rolling.merge(
    py_daily[["date", "py_rolling_vol_21d", "py_rolling_ann_vol_21d"]],
    on="date",
    how="inner",
)
joined_drawdowns = sql_drawdowns.merge(
    py_prices[["date", "py_running_peak", "py_drawdown"]],
    on="date",
    how="inner",
)

py_summary = {
    "annualized_return": py_daily["py_log_return"].mean() * 252,
    "annualized_volatility": py_daily["py_log_return"].std(ddof=1) * np.sqrt(252),
    "max_drawdown": py_prices["py_drawdown"].min(),
    "bad_day_rate_2pct": (py_daily["py_simple_return"] <= -0.02).mean(),
    "bad_day_rate_5pct": (py_daily["py_simple_return"] <= -0.05).mean(),
}

validation_results = pd.DataFrame(
    [
        {
            "check": "simple_return",
            "max_abs_diff": np.max(np.abs(joined_daily["simple_return"] - joined_daily["py_simple_return"])),
            "passed": np.allclose(joined_daily["simple_return"], joined_daily["py_simple_return"], atol=TOLERANCE, equal_nan=True),
        },
        {
            "check": "log_return",
            "max_abs_diff": np.max(np.abs(joined_daily["log_return"] - joined_daily["py_log_return"])),
            "passed": np.allclose(joined_daily["log_return"], joined_daily["py_log_return"], atol=TOLERANCE, equal_nan=True),
        },
        {
            "check": "dollar_volume",
            "max_abs_diff": np.max(np.abs(joined_daily["dollar_volume"] - joined_daily["py_dollar_volume"])),
            "passed": np.allclose(joined_daily["dollar_volume"], joined_daily["py_dollar_volume"], atol=TOLERANCE, equal_nan=True),
        },
        {
            "check": "rolling_vol_21d",
            "max_abs_diff": np.nanmax(np.abs(joined_rolling["rolling_vol_21d"] - joined_rolling["py_rolling_vol_21d"])),
            "passed": np.allclose(joined_rolling["rolling_vol_21d"], joined_rolling["py_rolling_vol_21d"], atol=TOLERANCE, equal_nan=True),
        },
        {
            "check": "rolling_ann_vol_21d",
            "max_abs_diff": np.nanmax(np.abs(joined_rolling["rolling_ann_vol_21d"] - joined_rolling["py_rolling_ann_vol_21d"])),
            "passed": np.allclose(joined_rolling["rolling_ann_vol_21d"], joined_rolling["py_rolling_ann_vol_21d"], atol=TOLERANCE, equal_nan=True),
        },
        {
            "check": "running_peak",
            "max_abs_diff": np.max(np.abs(joined_drawdowns["running_peak"] - joined_drawdowns["py_running_peak"])),
            "passed": np.allclose(joined_drawdowns["running_peak"], joined_drawdowns["py_running_peak"], atol=TOLERANCE, equal_nan=True),
        },
        {
            "check": "drawdown",
            "max_abs_diff": np.max(np.abs(joined_drawdowns["drawdown"] - joined_drawdowns["py_drawdown"])),
            "passed": np.allclose(joined_drawdowns["drawdown"], joined_drawdowns["py_drawdown"], atol=TOLERANCE, equal_nan=True),
        },
        {
            "check": "annualized_return",
            "max_abs_diff": abs(sql_risk_summary.loc[0, "annualized_return"] - py_summary["annualized_return"]),
            "passed": np.isclose(sql_risk_summary.loc[0, "annualized_return"], py_summary["annualized_return"], atol=TOLERANCE),
        },
        {
            "check": "annualized_volatility",
            "max_abs_diff": abs(sql_risk_summary.loc[0, "annualized_volatility"] - py_summary["annualized_volatility"]),
            "passed": np.isclose(sql_risk_summary.loc[0, "annualized_volatility"], py_summary["annualized_volatility"], atol=TOLERANCE),
        },
        {
            "check": "max_drawdown",
            "max_abs_diff": abs(sql_risk_summary.loc[0, "max_drawdown"] - py_summary["max_drawdown"]),
            "passed": np.isclose(sql_risk_summary.loc[0, "max_drawdown"], py_summary["max_drawdown"], atol=TOLERANCE),
        },
        {
            "check": "bad_day_rate_2pct",
            "max_abs_diff": abs(sql_risk_summary.loc[0, "bad_day_rate_2pct"] - py_summary["bad_day_rate_2pct"]),
            "passed": np.isclose(sql_risk_summary.loc[0, "bad_day_rate_2pct"], py_summary["bad_day_rate_2pct"], atol=TOLERANCE),
        },
        {
            "check": "bad_day_rate_5pct",
            "max_abs_diff": abs(sql_risk_summary.loc[0, "bad_day_rate_5pct"] - py_summary["bad_day_rate_5pct"]),
            "passed": np.isclose(sql_risk_summary.loc[0, "bad_day_rate_5pct"], py_summary["bad_day_rate_5pct"], atol=TOLERANCE),
        },
    ]
)

validation_results


In [ ]:
if not validation_results["passed"].all():
    failed_checks = validation_results.loc[~validation_results["passed"], "check"].tolist()
    raise AssertionError(f"SQL validation failed for: {failed_checks}")

print(f"All SQL feature checks passed for {SYMBOL_TO_VALIDATE}.")


## 7. Save the processed modeling dataset

The modeling dataset combines the row-level rolling feature table with drawdown and downside-indicator columns. This keeps one observation per `symbol` / `date` and preserves the main predictors needed by downstream Bayesian return, volatility, downside-risk, regime, and portfolio notebooks.


In [ ]:
PROCESSED_FEATURE_PATH.parent.mkdir(parents=True, exist_ok=True)

modeling_features = query_to_df(
    con,
    """
    SELECT
        r.symbol,
        r.date,
        r.adj_close,
        r.volume,
        r.simple_return,
        r.log_return,
        r.dollar_volume,
        r.return_21d,
        r.return_63d,
        r.return_126d,
        r.return_252d,
        r.rolling_mean_21d,
        r.rolling_vol_21d,
        r.rolling_vol_63d,
        r.rolling_vol_126d,
        r.rolling_vol_252d,
        r.rolling_ann_vol_21d,
        r.rolling_ann_vol_63d,
        r.rolling_ann_vol_126d,
        r.rolling_ann_vol_252d,
        r.ma_20,
        r.ma_50,
        r.ma_200,
        r.above_ma_20,
        r.above_ma_50,
        r.above_ma_200,
        r.avg_volume_21d,
        r.avg_volume_63d,
        r.avg_dollar_volume_21d,
        r.volume_zscore_63d,
        d.running_peak,
        d.drawdown,
        CASE WHEN r.simple_return <= -0.02 THEN 1 ELSE 0 END AS bad_day_2pct,
        CASE WHEN r.simple_return <= -0.05 THEN 1 ELSE 0 END AS bad_day_5pct,
        EXTRACT(YEAR FROM r.date)::INTEGER AS year,
        EXTRACT(MONTH FROM r.date)::INTEGER AS month
    FROM rolling_features AS r
    INNER JOIN drawdowns AS d
        ON r.symbol = d.symbol
       AND r.date = d.date
    ORDER BY r.symbol, r.date
    """,
)

modeling_features.to_csv(PROCESSED_FEATURE_PATH, index=False)

print(f"Saved {len(modeling_features):,} rows and {modeling_features.shape[1]} columns to {PROCESSED_FEATURE_PATH}")
modeling_features.head()


## 8. Conclusion

- **SQL created the core analytical layer.** The pipeline transformed raw OHLCV records into cleaned prices, returns, rolling features, drawdowns, risk summaries, annual metrics, and portfolio inputs.
- **These tables feed the Bayesian models.** The downstream notebooks can use `daily_returns`, `rolling_features`, `stock_level_risk_summary`, `annual_stock_metrics`, `portfolio_returns_wide`, and the saved `data/processed/tech_stock_features.csv` as model-ready inputs.
- **Adjusted close is used for return calculations.** This is essential because return and risk measures should reflect corporate actions such as splits.
- **Rolling and downside-risk features help describe risk dynamics.** Rolling volatility, moving-average state, drawdown, and bad-day indicators capture time-varying risk that static return summaries can miss.

Overall, this SQL layer makes the modeling workflow more reproducible, auditable, and scalable than ad hoc notebook-only transformations.
